# Phase 3: GraphGPS Classifier

Trains `GPSClassifier` on Twitter15 and Twitter16 using 5 seeds.
Compares against GCN/GAT baselines from Phase 2.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import os
import torch
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

os.makedirs("../results", exist_ok=True)

from data.dataset import CascadeDataset
from models.gps import GPSClassifier
from training.trainer import run_experiment

In [ ]:
DATA_ROOT = "../Twitter15_16_dataset-main"
SEEDS = [0, 1, 2, 3, 4]

if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

In [ ]:
ds15 = CascadeDataset(root=DATA_ROOT, name="twitter15")
ds16 = CascadeDataset(root=DATA_ROOT, name="twitter16")
print(f"Twitter15: {len(ds15)} graphs, x={ds15[0].x.shape}, edge_attr={ds15[0].edge_attr.shape}")
print(f"Twitter16: {len(ds16)} graphs, x={ds16[0].x.shape}, edge_attr={ds16[0].edge_attr.shape}")

IN_CHANNELS = ds15[0].x.shape[1]  # 30
EDGE_DIM = ds15[0].edge_attr.shape[1]  # 2
print(f"in_channels={IN_CHANNELS}, edge_dim={EDGE_DIM}")

In [ ]:
GPS_KWARGS = dict(
    in_channels=IN_CHANNELS,
    hidden_channels=128,
    num_layers=3,
    heads=4,
    dropout=0.1,
    edge_dim=EDGE_DIM,
)

In [ ]:
print("=" * 50)
print("GraphGPS — Twitter15")
print("=" * 50)
res15 = run_experiment(
    GPSClassifier, GPS_KWARGS, ds15, SEEDS,
    epochs=200,
    lr=1e-3,
    weight_decay=0.05,
    patience=40,
    warmup_ratio=0.1,
    lap_pe_sign_flip=True,
    max_nodes_per_batch=16384,  # ~40 avg-sized graphs/batch; caps N_max² OOM for large cascades
    device=DEVICE,
)
print(f"\nTwitter15 GPS — Acc: {res15['test_acc_mean']:.3f} ± {res15['test_acc_std']:.3f}  "
      f"F1: {res15['test_f1_mean']:.3f} ± {res15['test_f1_std']:.3f}")

with open("../results/gps_twitter15.json", "w") as f:
    json.dump(res15, f, indent=2)

In [ ]:
print("=" * 50)
print("GraphGPS — Twitter16")
print("=" * 50)
res16 = run_experiment(
    GPSClassifier, GPS_KWARGS, ds16, SEEDS,
    epochs=200,
    lr=1e-3,
    weight_decay=0.05,
    patience=40,
    warmup_ratio=0.1,
    lap_pe_sign_flip=True,
    max_nodes_per_batch=16384,
    device=DEVICE,
)
print(f"\nTwitter16 GPS — Acc: {res16['test_acc_mean']:.3f} ± {res16['test_acc_std']:.3f}  "
      f"F1: {res16['test_f1_mean']:.3f} ± {res16['test_f1_std']:.3f}")

with open("../results/gps_twitter16.json", "w") as f:
    json.dump(res16, f, indent=2)

In [ ]:
# Comparison table — load baselines from saved JSON
def load_result(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

gcn15 = load_result("../results/gcn_twitter15.json")
gat15 = load_result("../results/gat_twitter15.json")
gcn16 = load_result("../results/gcn_twitter16.json")
gat16 = load_result("../results/gat_twitter16.json")

rows = [
    ("GCN",  "Twitter15", gcn15),
    ("GAT",  "Twitter15", gat15),
    ("GCN",  "Twitter16", gcn16),
    ("GAT",  "Twitter16", gat16),
    ("GPS",  "Twitter15", res15),
    ("GPS",  "Twitter16", res16),
]

print()
print("=" * 65)
print(f"{'Model':<10} {'Dataset':<14} {'Acc mean±std':>16}  {'Macro-F1 mean±std':>18}")
print("-" * 65)
for model, ds_name, res in rows:
    if res is None:
        print(f"{model:<10} {ds_name:<14}  (no results yet — run 02_baselines.ipynb first)")
        continue
    a_m, a_s = res['test_acc_mean'], res['test_acc_std']
    f_m, f_s = res['test_f1_mean'], res['test_f1_std']
    print(f"{model:<10} {ds_name:<14} {a_m:.3f} ± {a_s:.3f}        {f_m:.3f} ± {f_s:.3f}")
print("=" * 65)
print("(5 seeds, 60/20/20 stratified split, best val-F1 checkpoint)")